# Unit Tests - Market Price Prediction Component
### Covers all 10 main functionalities across all 4 source files
| File | Functionalities Tested |
|---|---|
| `weekly_update.py` | Feature Engineering, Week & Date Calculation, Backend Extraction, Fuel Scraping |
| `final-model-training-all.ipynb` | Data Cleaning, Model Evaluation Metrics, TimeSeriesSplit |
| `Weekly-Tuning-final.ipynb` | Best Model Selection |
| `Market-Backend.ipynb` | Prediction Pipeline, Output Validation |

> **Note:** Make sure `weekly_update.py` is in the same folder as this notebook before running.

## Setup - Imports & Load Functions from weekly_update.py

In [1]:
import unittest
import numpy as np
import pandas as pd
from datetime import date, datetime
from unittest.mock import patch, MagicMock
import sys, os

sys.path.insert(0, os.getcwd())

from weekly_update import (
    get_custom_week,
    week_to_csv_monday,
    get_backend_data,
    build_row,
    VEGETABLES,
    VEG_CODE,
    BACKEND_VEG_MAP,
)

print("Imports successful")
print("Vegetables:", VEGETABLES)

Imports successful
Vegetables: ['Bitter Gourd', 'Brinjals', 'Cabbage', 'Carrot', 'Pumpkin', 'Tomatoes']


## Functionality 1 - Weekly Feature Engineering (`weekly_update.py`)
`build_row()` constructs all 63 features for one vegetable for one week.
Tests cover: price lags, rolling stats, price change, cyclical time encoding,
season assignment, vegetable one-hot encoding, and all external features.

In [2]:
class TestWeeklyFeatureEngineering(unittest.TestCase):

    def _backend(self):
        return {
            "USD_LKR": 302.5, "ExchangeRate_Change": 0.5,
            "avg_flood_prob": 0.12, "avg_drought_prob": 0.08,
            "avg_precipitation": 0.80,
            "wholesale": {v: 200 + i * 20 for i, v in enumerate(VEGETABLES)},
        }

    def _row(self, veg="Bitter Gourd", price=400, n=15):
        ph = [350 + i for i in range(n)]
        return build_row(
            veg, price, pd.Timestamp("2026-03-09"), 290.0,
            self._backend(), ph, [290.0] * n, [180.0] * n, {3: 370.0}
        )

    def test_all_required_columns_present(self):
        row = self._row()
        required = [
            "Date", "Vegetable", "Price", "Month", "Quarter",
            "Price_Lag_1", "Price_Lag_2", "Price_Lag_3", "Price_Lag_4",
            "Price_Lag_8", "Price_Lag_12", "Price_Lag_52",
            "Rolling_Mean_4", "Rolling_Mean_8", "Rolling_Mean_12",
            "Rolling_Std_4", "Rolling_Min_4", "Rolling_Max_4",
            "Price_Change_1wk", "Price_Change_Pct_1wk",
            "Price_Change_4wk", "Price_Change_12wk",
            "Fuel_Price", "Fuel_Lag_1", "Fuel_Lag_2", "Fuel_Lag_3", "Fuel_Lag_4",
            "Fuel_Rolling_Mean_4",
            "Month_Sin", "Month_Cos", "Week_Sin", "Week_Cos", "Season",
            "USD_LKR", "ExchangeRate_Change",
            "avg_flood_prob", "avg_drought_prob", "avg_precipitation",
            "Wholesale_Price", "Wholesale_Lag1", "Wholesale_Lag2",
            "Wholesale_Rolling_Mean4", "Vegetable_Code",
        ]
        for col in required:
            self.assertIn(col, row, f"Missing feature column: {col}")

    def test_price_lag1_is_last_history_value(self):
        n = 15
        ph = [350 + i for i in range(n)]
        row = build_row("Bitter Gourd", 400, pd.Timestamp("2026-03-09"),
                        290.0, self._backend(), ph, [290.0]*n, [180.0]*n, {3: 370.0})
        self.assertAlmostEqual(row["Price_Lag_1"], ph[n - 1])

    def test_price_lag2_is_second_to_last(self):
        n = 15
        ph = [350 + i for i in range(n)]
        row = build_row("Bitter Gourd", 400, pd.Timestamp("2026-03-09"),
                        290.0, self._backend(), ph, [290.0]*n, [180.0]*n, {3: 370.0})
        self.assertAlmostEqual(row["Price_Lag_2"], ph[n - 2])

    def test_price_lag52_is_nan_when_history_too_short(self):
        self.assertTrue(np.isnan(self._row(n=10)["Price_Lag_52"]))

    def test_all_lags_nan_when_no_history(self):
        row = build_row("Bitter Gourd", 400, pd.Timestamp("2026-03-09"),
                        290.0, self._backend(), [], [], [], {})
        self.assertTrue(np.isnan(row["Price_Lag_1"]))
        self.assertTrue(np.isnan(row["Rolling_Mean_4"]))

    def test_rolling_mean_4_is_average_of_last_4(self):
        n = 15
        ph = [350 + i for i in range(n)]
        self.assertAlmostEqual(self._row(n=n)["Rolling_Mean_4"], np.mean(ph[-4:]))

    def test_rolling_min_4_is_minimum_of_last_4(self):
        n = 15
        ph = [350 + i for i in range(n)]
        self.assertAlmostEqual(self._row(n=n)["Rolling_Min_4"], np.min(ph[-4:]))

    def test_rolling_max_4_is_maximum_of_last_4(self):
        n = 15
        ph = [350 + i for i in range(n)]
        self.assertAlmostEqual(self._row(n=n)["Rolling_Max_4"], np.max(ph[-4:]))

    def test_price_change_1wk_is_current_minus_lag1(self):
        n = 15
        ph = [350 + i for i in range(n)]
        self.assertAlmostEqual(self._row(price=400, n=n)["Price_Change_1wk"], 400 - ph[n - 1])

    def test_month_sin_within_valid_range(self):
        self.assertTrue(-1 <= self._row()["Month_Sin"] <= 1)

    def test_month_cos_within_valid_range(self):
        self.assertTrue(-1 <= self._row()["Month_Cos"] <= 1)

    def test_week_sin_within_valid_range(self):
        self.assertTrue(-1 <= self._row()["Week_Sin"] <= 1)

    def test_march_is_season_2(self):
        self.assertEqual(self._row()["Season"], 2)

    def test_vegetable_code_matches_veg_code_dict(self):
        for veg in VEGETABLES:
            self.assertEqual(self._row(veg=veg)["Vegetable_Code"], VEG_CODE[veg])

    def test_one_hot_flag_is_1_for_correct_vegetable(self):
        self.assertEqual(self._row(veg="Cabbage")["Is_Cabbage"], 1)

    def test_one_hot_flag_is_0_for_other_vegetables(self):
        row = self._row(veg="Cabbage")
        self.assertEqual(row["Is_Tomatoes"], 0)
        self.assertEqual(row["Is_Carrot"], 0)

    def test_usd_lkr_taken_from_backend(self):
        self.assertAlmostEqual(self._row()["USD_LKR"], 302.5)

    def test_flood_prob_taken_from_backend(self):
        self.assertAlmostEqual(self._row()["avg_flood_prob"], 0.12)

    def test_wholesale_price_taken_from_backend(self):
        row = self._row(veg="Bitter Gourd")
        self.assertAlmostEqual(row["Wholesale_Price"],
                               self._backend()["wholesale"]["Bitter Gourd"])

    def test_fuel_price_stored_correctly(self):
        self.assertEqual(self._row()["Fuel_Price"], 290.0)


suite = unittest.TestLoader().loadTestsFromTestCase(TestWeeklyFeatureEngineering)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\nALL PASSED: {result.wasSuccessful()} | Tests run: {result.testsRun}")

test_all_lags_nan_when_no_history (__main__.TestWeeklyFeatureEngineering.test_all_lags_nan_when_no_history) ... 

ok


test_all_required_columns_present (__main__.TestWeeklyFeatureEngineering.test_all_required_columns_present) ... 

ok


test_flood_prob_taken_from_backend (__main__.TestWeeklyFeatureEngineering.test_flood_prob_taken_from_backend) ... 

...

ok


test_wholesale_price_taken_from_backend (__main__.TestWeeklyFeatureEngineering.test_wholesale_price_taken_from_backend) ... 

ok


----------------------------------------------------------------------
Ran 20 tests in 0.086s

OK



ALL PASSED: True | Tests run: 20


## Functionality 2 - Custom Week & Date Calculation (`weekly_update.py`)
`get_custom_week()` maps any date to the correct week number (1-52).
`week_to_csv_monday()` derives the Monday date stored in the CSV.
Wrong week number = data appended to wrong row in the dataset.

In [3]:
class TestWeekAndDateCalculation(unittest.TestCase):

    def test_jan_1_is_week_1(self):
        self.assertEqual(get_custom_week(date(2026, 1, 1)), 1)

    def test_jan_7_is_still_week_1(self):
        self.assertEqual(get_custom_week(date(2026, 1, 7)), 1)

    def test_jan_8_starts_week_2(self):
        self.assertEqual(get_custom_week(date(2026, 1, 8)), 2)

    def test_week_9_covers_feb26_to_mar4(self):
        self.assertEqual(get_custom_week(date(2026, 2, 26)), 9)
        self.assertEqual(get_custom_week(date(2026, 3, 4)),  9)

    def test_dec_31_is_week_52(self):
        self.assertEqual(get_custom_week(date(2026, 12, 31)), 52)

    def test_week_number_always_at_least_1(self):
        for m in range(1, 13):
            self.assertGreaterEqual(get_custom_week(date(2026, m, 1)), 1)

    def test_csv_monday_is_a_monday(self):
        self.assertEqual(week_to_csv_monday(2026, 9).weekday(), 0)

    def test_week_9_csv_monday_is_feb_23(self):
        self.assertEqual(week_to_csv_monday(2026, 9), date(2026, 2, 23))

    def test_csv_monday_returns_date_object(self):
        self.assertIsInstance(week_to_csv_monday(2026, 5), date)


suite = unittest.TestLoader().loadTestsFromTestCase(TestWeekAndDateCalculation)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\nALL PASSED: {result.wasSuccessful()} | Tests run: {result.testsRun}")

test_csv_monday_is_a_monday (__main__.TestWeekAndDateCalculation.test_csv_monday_is_a_monday) ... 

ok


test_csv_monday_returns_date_object (__main__.TestWeekAndDateCalculation.test_csv_monday_returns_date_object) ... 

ok


test_dec_31_is_week_52 (__main__.TestWeekAndDateCalculation.test_dec_31_is_week_52) ... 

...

ok


test_week_number_always_at_least_1 (__main__.TestWeekAndDateCalculation.test_week_number_always_at_least_1) ... 

ok


----------------------------------------------------------------------
Ran 9 tests in 0.011s

OK



ALL PASSED: True | Tests run: 9


## Functionality 3 - Backend Data Extraction (`weekly_update.py` + `Market-Backend.ipynb`)
`get_backend_data()` pulls USD/LKR, flood/drought probabilities, and per-vegetable
wholesale prices from `Backend_Data.csv` for the current week.
Missing data halts the entire pipeline.

In [4]:
class TestBackendDataExtraction(unittest.TestCase):

    def _make_df(self, year=2026, week=9):
        rows = []
        for vid, veg in BACKEND_VEG_MAP.items():
            rows.append({
                "year": year, "week_num": week, "vegetable": vid,
                "price": 200 + vid * 15,
                "USD_LKR_avg": 302.50, "RateChange_avg": 0.5,
                "avg_prob_flood_risk": 0.12, "avg_prob_drought": 0.08,
                "avg_prob_normal": 0.80,
            })
        return pd.DataFrame(rows)

    def test_usd_lkr_extracted_correctly(self):
        self.assertAlmostEqual(get_backend_data(self._make_df(), 2026, 9)["USD_LKR"], 302.50)

    def test_flood_prob_extracted_correctly(self):
        self.assertAlmostEqual(get_backend_data(self._make_df(), 2026, 9)["avg_flood_prob"], 0.12)

    def test_drought_prob_extracted_correctly(self):
        self.assertAlmostEqual(get_backend_data(self._make_df(), 2026, 9)["avg_drought_prob"], 0.08)

    def test_all_six_vegetables_have_wholesale_price(self):
        result = get_backend_data(self._make_df(), 2026, 9)
        for veg in VEGETABLES:
            self.assertIn(veg, result["wholesale"])

    def test_missing_week_returns_empty_dict(self):
        self.assertEqual(get_backend_data(self._make_df(week=9), 2026, 99), {})

    def test_wrong_year_returns_empty_dict(self):
        self.assertEqual(get_backend_data(self._make_df(year=2026), 2025, 9), {})


suite = unittest.TestLoader().loadTestsFromTestCase(TestBackendDataExtraction)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\nALL PASSED: {result.wasSuccessful()} | Tests run: {result.testsRun}")

test_all_six_vegetables_have_wholesale_price (__main__.TestBackendDataExtraction.test_all_six_vegetables_have_wholesale_price) ... 

ok


test_drought_prob_extracted_correctly (__main__.TestBackendDataExtraction.test_drought_prob_extracted_correctly) ... 

ok


test_flood_prob_extracted_correctly (__main__.TestBackendDataExtraction.test_flood_prob_extracted_correctly) ... 

...

ok


test_wrong_year_returns_empty_dict (__main__.TestBackendDataExtraction.test_wrong_year_returns_empty_dict) ... 

ok


----------------------------------------------------------------------
Ran 6 tests in 0.083s

OK


  [backend] W9  USD=302.50  flood=0.1200  drought=0.0800
  [backend] wholesale: {'Bitter Gourd': 215.0, 'Brinjals': 230.0, 'Cabbage': 245.0, 'Carrot': 260.0, 'Pumpkin': 275.0, 'Tomatoes': 290.0}
  [backend] W9  USD=302.50  flood=0.1200  drought=0.0800
  [backend] wholesale: {'Bitter Gourd': 215.0, 'Brinjals': 230.0, 'Cabbage': 245.0, 'Carrot': 260.0, 'Pumpkin': 275.0, 'Tomatoes': 290.0}
  [backend] W9  USD=302.50  flood=0.1200  drought=0.0800
  [backend] wholesale: {'Bitter Gourd': 215.0, 'Brinjals': 230.0, 'Cabbage': 245.0, 'Carrot': 260.0, 'Pumpkin': 275.0, 'Tomatoes': 290.0}
  [backend] W9  USD=302.50  flood=0.1200  drought=0.0800
  [backend] wholesale: {'Bitter Gourd': 215.0, 'Brinjals': 230.0, 'Cabbage': 245.0, 'Carrot': 260.0, 'Pumpkin': 275.0, 'Tomatoes': 290.0}

ALL PASSED: True | Tests run: 6


## Functionality 4 - Fuel Price Scraping (`weekly_update.py`)
`scrape_lad_price()` scrapes CEYPETCO for the current LAD (Lanka Auto Diesel) price.
Fuel cost is a feature in the model - wrong fuel price = wrong prediction.
Tests use a **mock HTTP response** so no real network call is made.

In [5]:
MOCK_HTML = """
<html><body><table>
  <tr><th>Date</th><th>LP95</th><th>LP92</th><th>LAD</th></tr>
  <tr><td>01.03.2026</td><td>340</td><td>320</td><td>295</td></tr>
  <tr><td>01.01.2026</td><td>340</td><td>320</td><td>285</td></tr>
</table></body></html>
"""

class TestFuelPriceScraping(unittest.TestCase):

    def _mock(self):
        r = MagicMock()
        r.text = MOCK_HTML
        r.raise_for_status = MagicMock()
        return r

    @patch("weekly_update.requests.get")
    def test_returns_lad_price_on_or_before_date(self, mg):
        mg.return_value = self._mock()
        from weekly_update import scrape_lad_price
        self.assertEqual(scrape_lad_price(date(2026, 3, 9)), 295.0)

    @patch("weekly_update.requests.get")
    def test_returns_older_revision_before_newer_one(self, mg):
        mg.return_value = self._mock()
        from weekly_update import scrape_lad_price
        self.assertEqual(scrape_lad_price(date(2026, 2, 1)), 285.0)

    @patch("weekly_update.requests.get")
    def test_raises_runtime_error_on_network_failure(self, mg):
        mg.side_effect = Exception("timeout")
        from weekly_update import scrape_lad_price
        with self.assertRaises(RuntimeError):
            scrape_lad_price(date(2026, 3, 9))


suite = unittest.TestLoader().loadTestsFromTestCase(TestFuelPriceScraping)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\nALL PASSED: {result.wasSuccessful()} | Tests run: {result.testsRun}")

test_raises_runtime_error_on_network_failure (__main__.TestFuelPriceScraping.test_raises_runtime_error_on_network_failure) ... 

ok


test_returns_lad_price_on_or_before_date (__main__.TestFuelPriceScraping.test_returns_lad_price_on_or_before_date) ... 

ok


test_returns_older_revision_before_newer_one (__main__.TestFuelPriceScraping.test_returns_older_revision_before_newer_one) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.010s

OK


  [fuel] Scraping CEYPETCO for LAD price ...
  [fuel] Scraping CEYPETCO for LAD price ...
  [fuel] LAD revision date 2026-03-01: LKR 295
  [fuel] Scraping CEYPETCO for LAD price ...
  [fuel] LAD revision date 2026-01-01: LKR 285

ALL PASSED: True | Tests run: 3


## Functionality 5 - Data Cleaning (`final-model-training-all.ipynb`)
Before training, the dataset is cleaned: `inf`/`-inf` replaced with `NaN`,
then all `NaN` rows dropped. Dirty data silently breaks model training.

In [6]:
class TestDataCleaning(unittest.TestCase):

    def _clean(self, df):
        df = df.copy()
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df = df.dropna().reset_index(drop=True)
        return df

    def test_inf_row_is_removed(self):
        self.assertEqual(len(self._clean(pd.DataFrame({"a": [1.0, np.inf, 3.0]}))), 2)

    def test_negative_inf_row_is_removed(self):
        self.assertEqual(len(self._clean(pd.DataFrame({"a": [1.0, -np.inf, 3.0]}))), 2)

    def test_nan_row_is_removed(self):
        self.assertEqual(len(self._clean(pd.DataFrame({"a": [1.0, np.nan, 3.0]}))), 2)

    def test_clean_data_is_not_changed(self):
        self.assertEqual(len(self._clean(pd.DataFrame({"a": [1.0, 2.0, 3.0]}))), 3)

    def test_no_inf_values_remain_after_cleaning(self):
        df = pd.DataFrame({"a": [np.inf, -np.inf, 1.0], "b": [1, 2, 3]})
        self.assertFalse(np.isinf(self._clean(df).values).any())

    def test_index_is_reset_after_cleaning(self):
        clean = self._clean(pd.DataFrame({"a": [np.nan, 2.0, 3.0]}))
        self.assertEqual(list(clean.index), list(range(len(clean))))


suite = unittest.TestLoader().loadTestsFromTestCase(TestDataCleaning)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\nALL PASSED: {result.wasSuccessful()} | Tests run: {result.testsRun}")

test_clean_data_is_not_changed (__main__.TestDataCleaning.test_clean_data_is_not_changed) ... 

ok

test_index_is_reset_after_cleaning (__main__.TestDataCleaning.test_index_is_reset_after_cleaning) ... 

ok


test_inf_row_is_removed (__main__.TestDataCleaning.test_inf_row_is_removed) ... 

...

ok


test_no_inf_values_remain_after_cleaning (__main__.TestDataCleaning.test_no_inf_values_remain_after_cleaning) ... 

ok


----------------------------------------------------------------------
Ran 6 tests in 0.019s

OK



ALL PASSED: True | Tests run: 6


## Functionality 6 - Model Evaluation Metrics (`final-model-training-all.ipynb`)
`evaluate_model()` computes **RMSE, MAE, R2, and MAPE** for each trained model.
These metrics compare XGBoost, LightGBM, and CatBoost to decide which gets deployed.
Uses dummy models with known outputs for precise verification.

In [7]:
class TestModelEvaluationMetrics(unittest.TestCase):

    def _evaluate(self, model, X_tr, y_tr, X_te, y_te):
        from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
        yp_tr = model.predict(X_tr)
        yp_te = model.predict(X_te)
        return {
            "Train RMSE": np.sqrt(mean_squared_error(y_tr, yp_tr)),
            "Test RMSE":  np.sqrt(mean_squared_error(y_te, yp_te)),
            "Test MAE":   mean_absolute_error(y_te, yp_te),
            "Test R2":    r2_score(y_te, yp_te),
            "Test MAPE":  np.mean(np.abs((y_te - yp_te) / y_te)) * 100,
        }

    class _Perfect:
        def predict(self, X): return np.array(X).flatten()

    class _Constant300:
        def predict(self, X): return np.full(len(X), 300.0)

    def _xy(self, vals):
        a = np.array(vals).reshape(-1, 1)
        return a, np.array(vals)

    def test_perfect_predictions_give_rmse_zero(self):
        X, y = self._xy([100, 200, 300])
        self.assertAlmostEqual(self._evaluate(self._Perfect(), X, y, X, y)["Test RMSE"], 0.0)

    def test_perfect_predictions_give_r2_one(self):
        X, y = self._xy([100, 200, 300])
        self.assertAlmostEqual(self._evaluate(self._Perfect(), X, y, X, y)["Test R2"], 1.0)

    def test_rmse_is_never_negative(self):
        X, y = self._xy([100, 200, 300])
        self.assertGreaterEqual(self._evaluate(self._Constant300(), X, y, X, y)["Test RMSE"], 0)

    def test_mae_is_never_negative(self):
        X, y = self._xy([100, 200, 300])
        self.assertGreaterEqual(self._evaluate(self._Constant300(), X, y, X, y)["Test MAE"], 0)

    def test_rmse_is_greater_or_equal_to_mae(self):
        X, y = self._xy([100, 200, 500])
        m = self._evaluate(self._Constant300(), X, y, X, y)
        self.assertGreaterEqual(m["Test RMSE"], m["Test MAE"])

    def test_all_five_metric_keys_are_returned(self):
        X, y = self._xy([100, 200])
        m = self._evaluate(self._Perfect(), X, y, X, y)
        for key in ["Train RMSE", "Test RMSE", "Test MAE", "Test R2", "Test MAPE"]:
            self.assertIn(key, m)


suite = unittest.TestLoader().loadTestsFromTestCase(TestModelEvaluationMetrics)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\nALL PASSED: {result.wasSuccessful()} | Tests run: {result.testsRun}")

test_all_five_metric_keys_are_returned (__main__.TestModelEvaluationMetrics.test_all_five_metric_keys_are_returned) ... 

ok


test_mae_is_never_negative (__main__.TestModelEvaluationMetrics.test_mae_is_never_negative) ... 

ok


test_perfect_predictions_give_r2_one (__main__.TestModelEvaluationMetrics.test_perfect_predictions_give_r2_one) ... 

...

ok


test_rmse_is_never_negative (__main__.TestModelEvaluationMetrics.test_rmse_is_never_negative) ... 

ok


----------------------------------------------------------------------
Ran 6 tests in 5.125s

OK



ALL PASSED: True | Tests run: 6


## Functionality 7 - TimeSeriesSplit / No Data Leakage (`final-model-training-all.ipynb` + `Weekly-Tuning-final.ipynb`)
Both notebooks use `TimeSeriesSplit(n_splits=10)` to split data.
For a time-series model, **test data must always come AFTER training data**.
Any index overlap = data leakage = artificially inflated accuracy.

In [8]:
class TestTimeSeriesSplitIntegrity(unittest.TestCase):

    def _last_split(self, n=100, splits=10):
        from sklearn.model_selection import TimeSeriesSplit
        X = pd.DataFrame({"a": range(n)})
        tscv = TimeSeriesSplit(n_splits=splits)
        for tr, te in tscv.split(X):
            pass
        return tr, te

    def test_test_data_is_always_after_train_data(self):
        tr, te = self._last_split()
        self.assertGreater(te.min(), tr.max())

    def test_no_index_overlap_between_train_and_test(self):
        tr, te = self._last_split()
        self.assertEqual(len(set(tr) & set(te)), 0)

    def test_train_set_is_larger_than_test_set(self):
        tr, te = self._last_split()
        self.assertGreater(len(tr), len(te))


suite = unittest.TestLoader().loadTestsFromTestCase(TestTimeSeriesSplitIntegrity)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\nALL PASSED: {result.wasSuccessful()} | Tests run: {result.testsRun}")

test_no_index_overlap_between_train_and_test (__main__.TestTimeSeriesSplitIntegrity.test_no_index_overlap_between_train_and_test) ... 

ok


test_test_data_is_always_after_train_data (__main__.TestTimeSeriesSplitIntegrity.test_test_data_is_always_after_train_data) ... 

ok


test_train_set_is_larger_than_test_set (__main__.TestTimeSeriesSplitIntegrity.test_train_set_is_larger_than_test_set) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.022s

OK



ALL PASSED: True | Tests run: 3


## Functionality 8 - Best Model Selection (`Weekly-Tuning-final.ipynb`)
After tuning all 3 models, each is ranked across all metrics.
The model with the best **average rank** is selected for deployment.
R2 is ranked as higher-is-better (opposite direction to RMSE/MAE/MAPE).

In [9]:
class TestBestModelSelection(unittest.TestCase):

    def _select(self, xgb_m, lgb_m, cat_m):
        comp = pd.DataFrame([xgb_m, lgb_m, cat_m],
                            index=["XGBoost", "LightGBM", "CatBoost"])
        ranks = comp.rank(axis=0, ascending=[True, True, True, False, True])
        return ranks.mean(axis=1).idxmin()

    def _m(self, rmse, r2, mape):
        return {"Train RMSE": rmse, "Test RMSE": rmse,
                "Test MAE": rmse * 0.8, "Test R2": r2, "Test MAPE": mape}

    def test_clearly_best_model_is_selected(self):
        self.assertEqual(self._select(
            self._m(50, 0.60, 15),
            self._m(25, 0.90, 8),
            self._m(40, 0.70, 12),
        ), "LightGBM")

    def test_catboost_selected_when_it_wins(self):
        self.assertEqual(self._select(
            self._m(60, 0.50, 20),
            self._m(55, 0.55, 18),
            self._m(20, 0.95, 6),
        ), "CatBoost")

    def test_result_is_always_one_of_three_model_names(self):
        self.assertIn(self._select(
            self._m(35, 0.75, 11),
            self._m(33, 0.77, 10),
            self._m(34, 0.76, 11),
        ), ["XGBoost", "LightGBM", "CatBoost"])

    def test_r2_treated_as_higher_is_better(self):
        self.assertEqual(self._select(
            self._m(50, 0.50, 20),
            self._m(20, 0.95, 7),
            self._m(45, 0.55, 18),
        ), "LightGBM")


suite = unittest.TestLoader().loadTestsFromTestCase(TestBestModelSelection)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\nALL PASSED: {result.wasSuccessful()} | Tests run: {result.testsRun}")

test_catboost_selected_when_it_wins (__main__.TestBestModelSelection.test_catboost_selected_when_it_wins) ... 

ok


test_clearly_best_model_is_selected (__main__.TestBestModelSelection.test_clearly_best_model_is_selected) ... 

ok


test_r2_treated_as_higher_is_better (__main__.TestBestModelSelection.test_r2_treated_as_higher_is_better) ... 

ok


test_result_is_always_one_of_three_model_names (__main__.TestBestModelSelection.test_result_is_always_one_of_three_model_names) ... 

ok


----------------------------------------------------------------------
Ran 4 tests in 0.013s

OK



ALL PASSED: True | Tests run: 4


## Functionality 9 - Prediction Pipeline: Lag Rolling & Input Injection (`Market-Backend.ipynb`)
Before inference, the last known row is updated:
- **Price lag chain** shifts forward (Lag1 to Lag4) with the new price
- **Arosha's wholesale prediction** is injected with its own lag chain
- **Amika's weather prediction** (flood & drought probabilities) is injected

In [10]:
class TestPredictionPipeline(unittest.TestCase):

    def _last(self):
        return {
            "Price_Lag_1": 300.0, "Price_Lag_2": 290.0,
            "Price_Lag_3": 280.0, "Price_Lag_4": 270.0,
            "Rolling_Mean_4": 285.0, "Rolling_Mean_8": 280.0,
            "Wholesale_Price": 180.0, "Wholesale_Lag1": 170.0,
            "Wholesale_Lag2": 160.0,
            "avg_flood_prob": 0.0, "avg_drought_prob": 0.0,
        }

    def _roll(self, last, new_price):
        r = last.copy()
        r["Price_Lag_4"] = last["Price_Lag_3"]
        r["Price_Lag_3"] = last["Price_Lag_2"]
        r["Price_Lag_2"] = last["Price_Lag_1"]
        r["Price_Lag_1"] = new_price
        r["Rolling_Mean_4"] = (last["Price_Lag_1"] + last["Price_Lag_2"] +
                               last["Price_Lag_3"] + new_price) / 4
        r["Rolling_Mean_8"] = (r["Rolling_Mean_4"] + last["Rolling_Mean_4"]) / 2
        return r

    def _inject_ws(self, last, wp):
        r = last.copy()
        r["Wholesale_Price"]         = wp
        r["Wholesale_Lag1"]          = last["Wholesale_Price"]
        r["Wholesale_Lag2"]          = last["Wholesale_Lag1"]
        r["Wholesale_Rolling_Mean4"] = (wp + last["Wholesale_Price"] +
                                        last["Wholesale_Lag1"] + last["Wholesale_Lag2"]) / 4
        return r

    def test_lag1_becomes_the_new_price(self):
        self.assertEqual(self._roll(self._last(), 310)["Price_Lag_1"], 310)

    def test_lag2_becomes_previous_lag1(self):
        self.assertEqual(self._roll(self._last(), 310)["Price_Lag_2"], 300.0)

    def test_lag3_becomes_previous_lag2(self):
        self.assertEqual(self._roll(self._last(), 310)["Price_Lag_3"], 290.0)

    def test_lag4_becomes_previous_lag3(self):
        self.assertEqual(self._roll(self._last(), 310)["Price_Lag_4"], 280.0)

    def test_rolling_mean_4_recalculated_correctly(self):
        self.assertAlmostEqual(
            self._roll(self._last(), 310)["Rolling_Mean_4"],
            (300 + 290 + 280 + 310) / 4
        )

    def test_rolling_mean_8_is_average_of_both_mean4s(self):
        last = self._last()
        r = self._roll(last, 310)
        self.assertAlmostEqual(r["Rolling_Mean_8"],
                               (r["Rolling_Mean_4"] + last["Rolling_Mean_4"]) / 2)

    def test_wholesale_updated_to_arosha_prediction(self):
        self.assertEqual(self._inject_ws(self._last(), 200)["Wholesale_Price"], 200)

    def test_wholesale_lag1_is_previous_wholesale(self):
        self.assertEqual(self._inject_ws(self._last(), 200)["Wholesale_Lag1"], 180.0)

    def test_wholesale_rolling_mean4_correct(self):
        self.assertAlmostEqual(
            self._inject_ws(self._last(), 200)["Wholesale_Rolling_Mean4"],
            (200 + 180 + 170 + 160) / 4
        )

    def test_flood_prob_updated_from_amika(self):
        row = self._last()
        row["avg_flood_prob"] = 0.18
        self.assertAlmostEqual(row["avg_flood_prob"], 0.18)

    def test_drought_prob_updated_from_amika(self):
        row = self._last()
        row["avg_drought_prob"] = 0.07
        self.assertAlmostEqual(row["avg_drought_prob"], 0.07)


suite = unittest.TestLoader().loadTestsFromTestCase(TestPredictionPipeline)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\nALL PASSED: {result.wasSuccessful()} | Tests run: {result.testsRun}")

test_drought_prob_updated_from_amika (__main__.TestPredictionPipeline.test_drought_prob_updated_from_amika) ... 

ok


test_flood_prob_updated_from_amika (__main__.TestPredictionPipeline.test_flood_prob_updated_from_amika) ... 

ok


test_lag1_becomes_the_new_price (__main__.TestPredictionPipeline.test_lag1_becomes_the_new_price) ... 

...

ok


test_wholesale_updated_to_arosha_prediction (__main__.TestPredictionPipeline.test_wholesale_updated_to_arosha_prediction) ... 

ok


----------------------------------------------------------------------
Ran 11 tests in 0.014s

OK



ALL PASSED: True | Tests run: 11


## Functionality 10 - Prediction Output Validation (`Market-Backend.ipynb`)
The final output DataFrame must have the correct schema: one row per vegetable,
all required columns present, positive prices, and values rounded to 2 decimal places.

In [11]:
class TestPredictionOutputFormat(unittest.TestCase):

    def _output(self):
        return pd.DataFrame([
            {"Vegetable": v, "Week": "W9", "Year": 2026,
             "Predicted Price": round(300.0 + i * 10, 2),
             "Wholesale_Input": 180.0,
             "Flood_Risk": 0.12, "Drought_Risk": 0.08}
            for i, v in enumerate(VEGETABLES)
        ])

    def test_output_contains_all_six_vegetables(self):
        self.assertEqual(set(self._output()["Vegetable"]), set(VEGETABLES))

    def test_output_has_exactly_six_rows(self):
        self.assertEqual(len(self._output()), 6)

    def test_all_required_columns_present(self):
        df = self._output()
        for col in ["Vegetable", "Week", "Year", "Predicted Price",
                    "Wholesale_Input", "Flood_Risk", "Drought_Risk"]:
            self.assertIn(col, df.columns)

    def test_all_predicted_prices_are_positive(self):
        self.assertTrue((self._output()["Predicted Price"] > 0).all())

    def test_predicted_prices_rounded_to_2_decimal_places(self):
        for p in self._output()["Predicted Price"]:
            self.assertEqual(round(p, 2), p)

    def test_week_label_format_is_correct(self):
        self.assertTrue(self._output()["Week"].str.match(r"^W\d+$").all())


suite = unittest.TestLoader().loadTestsFromTestCase(TestPredictionOutputFormat)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\nALL PASSED: {result.wasSuccessful()} | Tests run: {result.testsRun}")

test_all_predicted_prices_are_positive (__main__.TestPredictionOutputFormat.test_all_predicted_prices_are_positive) ... 

ok


test_all_required_columns_present (__main__.TestPredictionOutputFormat.test_all_required_columns_present) ... 

ok


test_output_contains_all_six_vegetables (__main__.TestPredictionOutputFormat.test_output_contains_all_six_vegetables) ... 

...

ok


test_week_label_format_is_correct (__main__.TestPredictionOutputFormat.test_week_label_format_is_correct) ... 

ok


----------------------------------------------------------------------
Ran 6 tests in 0.012s

OK



ALL PASSED: True | Tests run: 6


## Full Suite Summary - Run All 74 Tests at Once

In [12]:
all_classes = [
    TestWeeklyFeatureEngineering,
    TestWeekAndDateCalculation,
    TestBackendDataExtraction,
    TestFuelPriceScraping,
    TestDataCleaning,
    TestModelEvaluationMetrics,
    TestTimeSeriesSplitIntegrity,
    TestBestModelSelection,
    TestPredictionPipeline,
    TestPredictionOutputFormat,
]

loader = unittest.TestLoader()
suite  = unittest.TestSuite()
for cls in all_classes:
    suite.addTests(loader.loadTestsFromTestCase(cls))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

print("\n" + "="*60)
print(f"  TOTAL TESTS RUN : {result.testsRun}")
print(f"  PASSED          : {result.testsRun - len(result.failures) - len(result.errors)}")
print(f"  FAILED          : {len(result.failures)}")
print(f"  ERRORS          : {len(result.errors)}")
print("="*60)
print(f"  RESULT: {'ALL TESTS PASSED' if result.wasSuccessful() else 'SOME TESTS FAILED'}")
print("="*60)

test_all_lags_nan_when_no_history (__main__.TestWeeklyFeatureEngineering.test_all_lags_nan_when_no_history) ... 

ok


test_all_required_columns_present (__main__.TestWeeklyFeatureEngineering.test_all_required_columns_present) ... 

ok


test_flood_prob_taken_from_backend (__main__.TestWeeklyFeatureEngineering.test_flood_prob_taken_from_backend) ... 

...

ok


test_week_label_format_is_correct (__main__.TestPredictionOutputFormat.test_week_label_format_is_correct) ... 

ok


----------------------------------------------------------------------
Ran 74 tests in 0.471s

OK



  TOTAL TESTS RUN : 74
  PASSED          : 74
  FAILED          : 0
  ERRORS          : 0
  RESULT: ALL TESTS PASSED
